# Patrón Creacional: Builder

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Builder** separa la **construcción** de un objeto complejo de su
**representación**, permitiendo crear el objeto **paso a paso**. Es ideal cuando un
objeto tiene **muchos campos**, varios de ellos **opcionales**.

### ¿Qué problema resuelve en la banca?
Abrir una cuenta bancaria implica reunir **muchos datos**: titular, documento, tipo de
cuenta, monto inicial, si incluye tarjeta débito, alertas SMS, sobregiro autorizado,
beneficiarios... Un constructor con 10 parámetros es ilegible y propenso a errores
(¿qué significaba el tercer `True`?). Builder arma la **solicitud de apertura** de forma
clara y encadenada.

## Código *sin patrón* (el problema es evidente)
Un único constructor gigante con muchos parámetros posicionales/opcionales.

In [1]:
class SolicitudCuentaSinPatron:
    def __init__(self, titular, documento, tipo="ahorros", monto_inicial=0,
                 tarjeta_debito=False, alertas_sms=False, sobregiro=0, beneficiario=None):
        self.titular = titular
        self.documento = documento
        self.tipo = tipo
        self.monto_inicial = monto_inicial
        self.tarjeta_debito = tarjeta_debito
        self.alertas_sms = alertas_sms
        self.sobregiro = sobregiro
        self.beneficiario = beneficiario

    def __str__(self):
        return (f"Cuenta {self.tipo} de {self.titular} | monto={self.monto_inicial} "
                f"| debito={self.tarjeta_debito} | sms={self.alertas_sms} "
                f"| sobregiro={self.sobregiro} | benef={self.beneficiario}")


# El que lee esta linea no sabe que significa cada valor sin ir al constructor
solicitud = SolicitudCuentaSinPatron("Ana Perez", "1020304050", "corriente", 500000,
                                     True, False, 200000, "Luis Perez")
print(solicitud)
print(">> Problema: constructor telescopico; los argumentos son ilegibles y faciles de confundir.")

Cuenta corriente de Ana Perez | monto=500000 | debito=True | sms=False | sobregiro=200000 | benef=Luis Perez
>> Problema: constructor telescopico; los argumentos son ilegibles y faciles de confundir.


### Análisis del problema
- El llamado `SolicitudCuentaSinPatron("Ana", ..., True, False, 200000, "Luis")` es
  **ilegible**: no se sabe qué significa cada `True/False` sin abrir el constructor.
- Agregar un nuevo dato opcional obliga a tocar la firma y **todas** las llamadas.
- Es fácil pasar los argumentos en el orden equivocado sin que nada falle.

## Código *con patrón* (problema resuelto)
Un `BuilderCuenta` construye la solicitud **paso a paso** con métodos legibles que
devuelven `self` (interfaz fluida). El método `construir()` entrega el producto final.

In [2]:
from dataclasses import dataclass, field


@dataclass
class SolicitudCuenta:
    titular: str
    documento: str
    tipo: str = "ahorros"
    monto_inicial: float = 0
    tarjeta_debito: bool = False
    alertas_sms: bool = False
    sobregiro: float = 0
    beneficiario: str = None

    def __str__(self):
        return (f"Cuenta {self.tipo} de {self.titular} | monto={self.monto_inicial} "
                f"| debito={self.tarjeta_debito} | sms={self.alertas_sms} "
                f"| sobregiro={self.sobregiro} | benef={self.beneficiario}")


class BuilderCuenta:
    def __init__(self, titular: str, documento: str):
        # Solo lo obligatorio se pide de entrada
        self._solicitud = SolicitudCuenta(titular=titular, documento=documento)

    def de_tipo(self, tipo: str):
        self._solicitud.tipo = tipo
        return self

    def con_monto_inicial(self, monto: float):
        self._solicitud.monto_inicial = monto
        return self

    def con_tarjeta_debito(self):
        self._solicitud.tarjeta_debito = True
        return self

    def con_alertas_sms(self):
        self._solicitud.alertas_sms = True
        return self

    def con_sobregiro(self, limite: float):
        self._solicitud.sobregiro = limite
        return self

    def con_beneficiario(self, nombre: str):
        self._solicitud.beneficiario = nombre
        return self

    def construir(self) -> SolicitudCuenta:
        return self._solicitud


# Construccion legible y autoexplicativa
solicitud = (
    BuilderCuenta("Ana Perez", "1020304050")
    .de_tipo("corriente")
    .con_monto_inicial(500000)
    .con_tarjeta_debito()
    .con_sobregiro(200000)
    .con_beneficiario("Luis Perez")
    .construir()
)
print(solicitud)

# Otra solicitud, mucho mas simple, sin repetir parametros que no aplican
cuenta_basica = BuilderCuenta("Carlos Ruiz", "9988776655").con_alertas_sms().construir()
print(cuenta_basica)
print(">> Solucion: cada paso se lee solo; los datos opcionales se agregan cuando aplican.")

Cuenta corriente de Ana Perez | monto=500000 | debito=True | sms=False | sobregiro=200000 | benef=Luis Perez
Cuenta ahorros de Carlos Ruiz | monto=0 | debito=False | sms=True | sobregiro=0 | benef=None
>> Solucion: cada paso se lee solo; los datos opcionales se agregan cuando aplican.


### Verificación
- Cada método (`.con_tarjeta_debito()`, `.con_sobregiro(...)`) **dice qué hace**.
- Una solicitud simple no arrastra parámetros que no aplican.
- Agregar un dato nuevo es sumar un método al builder, sin romper llamadas existentes.

## UML del patrón Builder
```plantuml
@startuml
class SolicitudCuenta {
    + titular
    + documento
    + tipo
    + monto_inicial
    + tarjeta_debito
    + alertas_sms
    + sobregiro
    + beneficiario
}
class BuilderCuenta {
    - _solicitud : SolicitudCuenta
    + de_tipo(tipo)
    + con_monto_inicial(monto)
    + con_tarjeta_debito()
    + con_alertas_sms()
    + con_sobregiro(limite)
    + con_beneficiario(nombre)
    + construir() : SolicitudCuenta
}
BuilderCuenta --> SolicitudCuenta : construye
@enduml
```

## ¿Por qué Builder y no otro patrón?
- El problema no es *elegir el tipo* de objeto (eso sería Factory) ni *tener una sola
  instancia* (Singleton): es **ensamblar un objeto con muchas partes opcionales** de
  forma legible y segura.
- Un Factory devolvería la cuenta pero seguiría necesitando pasar todos los datos de
  golpe. Builder brilla justamente cuando hay **construcción incremental** y muchas
  combinaciones válidas.
- Por eso Builder es la elección correcta para la **solicitud de apertura de cuenta**.